In [ ]:
import torch

from onomatron.data import Vocab, load_names, make_ngram_pairs, split_names
from onomatron.models.dilated_wavenet import DilatedWaveNet
from onomatron.utils import lr_range_test, plot_history, plot_lr_range, train

# Apple Silicon: use the integrated GPU via MPS when available (much faster
# than CPU for this model); fall back to CPU otherwise.
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

## Load names, split train/val/test, build vocab

In [ ]:
names = load_names("../data/processed/names.txt")
train_names, val_names, test_names = split_names(names, seed=42)
vocab = Vocab(train_names)  # vocab from train only: no leakage from val/test
print(
    f"{len(names)} names -> {len(train_names)} train / {len(val_names)} val"
    f" / {len(test_names)} test, vocab of {len(vocab)} chars"
)

In [ ]:
block_size = 8


def contexts(name_list: list[str]) -> tuple[torch.Tensor, torch.Tensor]:
    pairs = make_ngram_pairs(
        name_list, vocab, block_size + 1
    )  # (N, block_size+1), dernière col = cible
    return pairs[:, :-1], pairs[:, -1]


X, Y = contexts(train_names)
X_val, Y_val = contexts(val_names)
X_test, Y_test = contexts(test_names)
# Move everything to the device once: the training loop indexes these tensors
X, Y = X.to(device), Y.to(device)
X_val, Y_val = X_val.to(device), Y_val.to(device)
X_test, Y_test = X_test.to(device), Y_test.to(device)
print(f"{X.shape[0]} train / {X_val.shape[0]} val / {X_test.shape[0]} test contexts")

## Build DilatedWaveNet

Causal dilated convolutions (dilations 1, 2, 4, ...) instead of the
hierarchical fold of `wavenet.py`: the sequence keeps its length at every
layer and the receptive field of the last position is `2**n_layers`. The
model can predict from every position, but like the other notebooks we
train and evaluate on the last position only: all-positions training
overfits much faster on this small dataset.

In [ ]:
emb_dim, n_layers = 24, 3  # receptive field 2**3 = 8 = block_size

seed = 42
model = DilatedWaveNet(vocab, block_size, emb_dim, n_layers, seed).to(device)
print(f"DilatedWaveNet model has {model.n_parameters} parameters.")

## Find a maximum learning rate

In [ ]:
# Pick lr (minibatch, can take a while)
lrs = (10 ** torch.linspace(-3, 1, 100)).tolist()
losses = lr_range_test(
    lambda s: DilatedWaveNet(vocab, block_size, emb_dim, n_layers, seed=s).to(device),
    X,
    Y,
    lrs,
    batch_size=64,
)
best_lr = plot_lr_range(lrs, losses) / 10
print(f"Maximum learning rate found is {best_lr}")

## Train on train, watch val

In [ ]:
n_steps = 15000
batch_size = 64
lr = 0.001

model = DilatedWaveNet(vocab, block_size, emb_dim, n_layers, seed).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=best_lr, weight_decay=0.05)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=n_steps)

history = train(
    model,
    X,
    Y,
    opt,
    steps=n_steps,
    batch_size=batch_size,
    scheduler=sched,
    seed=42,
    x_val=X_val,
    y_val=Y_val,
    eval_every=100,
    clip_grad=1.0,
    patience=20,
)
plot_history(history, smooth=20)
print(
    f"Final train loss {history.train_losses[-1]:.4f}"
    f" | val NLL {history.val[-1][1]:.4f}"
)

## Sample from the model (min 3 chars)

In [ ]:
model.cpu()
model.sample(20)

## Final evaluation on test

In [ ]:
model.to(device)
print(
    f"DilatedWaveNet: train NLL {model.nll(X, Y):.4f}"
    f" | val NLL {model.nll(X_val, Y_val):.4f}"
    f" | test NLL {model.nll(X_test, Y_test):.4f}"
)